# Using the concept of Conditional Routing

# Importing Libraries

In [1]:
import joblib
import pandas as pd
from flask import Flask, request
from twilio.twiml.messaging_response import MessagingResponse
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict, Optional
from langchain_community.llms import Ollama

# Objective



# Model loading

In [2]:
model = joblib.load("Knn_model.pkl")

def predict_student(Study_Hours, Score):

    import pandas as pd
    input_data = pd.DataFrame([[Study_Hours, Score]], 
                               columns=['Study Hours', 'Previous Exam Score'])
    
    pred = model.predict(input_data)

    if pred[0] == 1:
        return "Pass"
    else:
        return "Fail"

# State graph

In [3]:
class State(TypedDict):
    input: str
    output: str
    current_mode: Optional[str] 
    user_name: Optional[str]
    education: Optional[str]
    study_hours: Optional[int]
    score: Optional[int]

# Node 1
- Router

In [4]:
def router_node(state):
    user_input = state.get("input", "").lower()
    
    prompt = f"""
    You are an intelligent routing assistant. Categorize the user's intent into exactly one of these two words:
    - 'prediction': If the user is asking to check their result, predict pass/fail, or analyze marks/scores.
    - 'chat': If the user is saying hello, greeting, asking general questions, or anything else.
    
    User Input: "{user_input}"
    Return ONLY 'prediction' or 'chat'. Do not add any other words.
    """
    
    decision = llm.invoke(prompt).strip().lower()
    
    # Decide mode based on intent
    if "prediction" in decision:
        return {"current_mode": "ml"}
    else:
        return {"current_mode": "llm"}

# Node 2
- LLM router

In [5]:
from langchain_ollama import OllamaLLM
llm = OllamaLLM(model="llama3.2")

def llm_node(state):
    user_input = state.get("input", "").strip()
    
    if user_input.lower() in ["menu", "exit", "quit", "back"]:
        return {"current_mode": None, "output": "Conversation reset. How can I help you today?"}
    
    try:
        response = llm.invoke(user_input)
        bot_msg = response if response else "I'm thinking... could you repeat that?"
        
        # Reply dene ke baad mode None kar do, taaki agla sawal wapas router ke paas jaye
        return {"output": bot_msg, "current_mode": None}
    except Exception as e:
        return {"output": f"Oops! LLM error: {e}", "current_mode": None}

# Node 3
- ML Model router

In [6]:
def ml_node(state):
    user_input = state.get("input", "").strip()
    user_input_lower = user_input.lower()
    raw_input = user_input
    
    # 🟢 Exit ML mode at any time
    if user_input_lower in ["menu", "exit", "quit", "cancel"]:
        return {
            "current_mode": None, 
            "user_name": None, "education": None, "study_hours": None, "score": None,
            "output": "Prediction cancelled. What would you like to talk about?"
        }

    user_name = state.get("user_name")
    education = state.get("education")
    study_hours = state.get("study_hours")
    score = state.get("score")

    if user_name is None:
        
        return {"user_name": raw_input, "output": "📊 Let's check the result! What is the student's name?"}

    if education is None:
        if user_input == "1": edu_level = "School"
        elif user_input == "2": edu_level = "College"
        elif user_input == "3": edu_level = "Competition Exam"
        else:
            return {"output": f"Nice to meet you, {user_name}! 🤝\nKis level ki padhai kar rahe hain?\n1️⃣ School\n2️⃣ College\n3️⃣ Competition Exam"}
            
        return {"education": edu_level, "output": f"Great! So preparing for {edu_level}. 📚\nDin mein kitne hours padhte ho? (Sirf number likhein)"}

    if study_hours is None:
        try:
            hours = int(user_input)
            return {"study_hours": hours, "output": "Perfect 👍\nAb batao average score (out of 100) kitna rehta hai?"}
        except ValueError:
            return {"output": "Kripya hours ko sirf number me batao (jaise: 5)"}

    if score is None:
        try:
            sc = int(user_input)
            result = predict_student(study_hours, sc)
            
            final_msg = (f"✨ Result for {user_name} ({education}) ✨\n\n"
                         f"🎯 AI Prediction: Aapke '{result}' hone ke chances hain!\n\n"
                         f"(Aap koi aur sawal pooch sakte hain ya nayi prediction start kar sakte hain)")
            
            # Form complete. Reset sab kuch.
            return {
                "user_name": None, "education": None, "study_hours": None, "score": None,
                "current_mode": None, 
                "output": final_msg
            }
        except ValueError:
            return {"output": "Kripya score ko sirf number me batao (jaise: 85)"}

# Graph Construction

In [10]:
def entry_router(state):
    # Agar user pehle se ML prediction form bhar raha hai toh wahi bhejo
    if state.get("current_mode") == "ml":
        return "ml_node"
    # Warna naya intent check karne router ke paas 
    return "router_node"

def route_after_router(state):
    # Router ne jo decide kiya, wahan bhejo
    if state.get("current_mode") == "ml":
        return "ml_node"
    return "llm_node"

memory = MemorySaver()
graph = StateGraph(State)

graph.add_node("router_node", router_node)
graph.add_node("llm_node", llm_node)
graph.add_node("ml_node", ml_node)

# Entry Point Decide karo
graph.set_conditional_entry_point(
    entry_router,
    {"ml_node": "ml_node", "router_node": "router_node"}
)

#  Router decide karne ke baad kahan jayega
graph.add_conditional_edges(
    "router_node",
    route_after_router,
    {"ml_node": "ml_node",
     "llm_node": "llm_node"}
)

graph.add_edge("llm_node", END)
graph.add_edge("ml_node", END)

graph = graph.compile(checkpointer=memory)

# Chatbot

In [11]:
config = {"configurable": {"thread_id": "smart_student_session_1"}}

print("--- 🤖 AI Smart Assistant Started ---")
print("Aap general sawal pooch sakte hain ya prediction ke liye keh sakte hain.")
print("Type 'exit' to stop the chatbot.\n")

while True:
    
    user_input = input("You: ").strip()
    
    
    if user_input.lower() in ['exit', 'quit']:
        print("Bot: Bye! Have a great day! 👋")
        break
    
    
    if not user_input:
        continue
        
    try:
        
        result = graph.invoke({"input": user_input}, config=config)
        
        
        print(f"Bot: {result['output']}\n")
        
    except Exception as e:
        print(f"Bot: 🚨 Oops! Kuch error aayi hai: {e}\n")

--- 🤖 AI Smart Assistant Started ---
Aap general sawal pooch sakte hain ya prediction ke liye keh sakte hain.
Type 'exit' to stop the chatbot.



You:  hi


Bot: How can I assist you today?



You:  what is ml in one line


Bot: Machine Learning (ML) is a subset of Artificial Intelligence (AI) that enables systems to automatically learn and improve from data, without being explicitly programmed.



...

You:  80


Bot: ✨ Result for i wont to predict wether i am fail or pass (College) ✨

🎯 AI Prediction: Aapke 'Pass' hone ke chances hain!

(Aap koi aur sawal pooch sakte hain ya nayi prediction start kar sakte hain)



You:  bye


Bot: Bye! Have a great day!



KeyboardInterrupt: Interrupted by user

# flask server

In [ ]:
from flask import Flask, request
from twilio.twiml.messaging_response import MessagingResponse

app = Flask(__name__)

@app.route("/whatsapp", methods=["POST"])
def whatsapp_reply():
    incoming_msg = request.form.get('Body').strip()
    sender_number = request.form.get('From')
    
    # Har user ka number thread_id banega
    config = {"configurable": {"thread_id": sender_number}}
    
    try:
        # Agent ko run karna
        result = graph.invoke({"input": incoming_msg}, config=config)
        
        # 🟢 SAFETY CHANGE: Agar 'output' key miss ho jaye toh server crash na ho
        bot_response = result.get("output", "🤖 Processing your request... please try again.")
        
    except Exception as e:
        bot_response = f"Sorry, mere system mein kuch gadbad ho gayi: {e}"
        print(f"🚨 Error: {e}")

    # JUPYTER KE LIYE PRINT STATEMENTS
    print(f"\n👉 User ne kaha: {incoming_msg}")
    print(f"👉 Bot ka Reply: {bot_response}\n")

    # Twilio ko wapas reply bhejna
    resp = MessagingResponse()
    resp.message(bot_response)
    
    return str(resp)

if __name__ == "__main__":
    # JUPYTER MEIN DEBUG=FALSE RAKHNA CHAHIYE
    print("🚀 Jupyter mein Flask Server Start ho raha hai...")
    app.run(port=5000, debug=False, use_reloader=False)

🚀 Jupyter mein Flask Server Start ho raha hai...
 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [23/Mar/2026 16:07:55] "POST /whatsapp HTTP/1.1" 200 -



👉 User ne kaha: Hii
👉 Bot ka Reply: Hello! How can I assist you today?



127.0.0.1 - - [23/Mar/2026 16:13:28] "POST /whatsapp HTTP/1.1" 200 -



👉 User ne kaha: Hi
👉 Bot ka Reply: How can I assist you today?



...


👉 User ne kaha: 8
👉 Bot ka Reply: Perfect 👍
Ab batao average score (out of 100) kitna rehta hai?



127.0.0.1 - - [23/Mar/2026 16:18:57] "POST /whatsapp HTTP/1.1" 200 -



👉 User ne kaha: 89
👉 Bot ka Reply: ✨ Result for Student pass fail prediction (College) ✨

🎯 AI Prediction: Aapke 'Pass' hone ke chances hain!

(Aap koi aur sawal pooch sakte hain ya nayi prediction start kar sakte hain)



127.0.0.1 - - [23/Mar/2026 16:19:10] "POST /whatsapp HTTP/1.1" 200 -



👉 User ne kaha: bye
👉 Bot ka Reply: Bye! Have a great day!



In [ ]:
ngrok http 5000